# Time Series Regression and SARIMAX Modeling on Electricity Demand

This notebook explores advanced time series forecasting techniques, specifically focusing on **regression models with exogenous variables** and the integration of **Fourier series for capturing complex seasonality**. We will apply these methods to the London Smart Meter dataset to forecast electricity demand.

The goal is to demonstrate how to:

1. Use a simple Linear Regression as a baseline, incorporating external factors like temperature.

2. Evaluate model performance using various metrics and cross-validation.

3. Enhance the regression model by adding Fourier series to explicitly capture intricate seasonal patterns.

4. Understand the concept of **Dynamic Regression (SARIMAX)**, where an ARIMA model is used to capture the remaining autocorrelation in the residuals of a regression model.

By the end of this notebook, you'll have a deeper understanding of how to build robust time series models that leverage both external factors and sophisticated seasonal components.

## 1. Setting Up the Environment: Imports and Configuration

We begin by importing all necessary libraries and setting up some global configurations for our plots and metrics.

### 1.1. Import Required Libraries

This section imports a comprehensive set of libraries essential for data manipulation, visualization, statistical modeling, and time series forecasting.

* **`polars`**: A highly performant DataFrame library, used for efficient data loading and manipulation.

* **`pandas`**: Another fundamental data manipulation library, often used for compatibility with other libraries.

* **`numpy`**: For numerical operations, especially array manipulation.

* **`matplotlib.pyplot`**: A basic plotting library, though we'll primarily use Plotly for interactive visualizations.

* **`plotly.graph_objects`, `plotly.express`, `plotly.io`**: For creating interactive and visually appealing plots. `plotly.io` is used to set a default template.

* **`statsforecast`**: A fast and scalable library for various time series models, including `AutoARIMA` and `MSTL`.

* **`statsmodels.stats.diagnostic.acorr_ljungbox`**: For the Ljung-Box test, used to check for autocorrelation in residuals.

* **`utilsforecast.evaluation` and `utilsforecast.losses`**: Essential for evaluating forecast accuracy using metrics like MAE, MSE, RMSE, MAPE, SMAPE, and MASE. `partial` from `functools` is used to pre-configure `mase` with a seasonality parameter.

* **`plotting_utils` (custom)**: A custom module containing helper functions for plotting time series, residuals, and comparison plots. **(Ensure `plotting_utils.py` is in your working directory or Python path.)**

* **`summary_utils` (custom)**: A custom module for printing summaries of fitted ARIMA and regression models, and for extracting residuals. **(Ensure `summary_utils.py` is in your working directory or Python path.)**

* **`prophet`**: Facebook's forecasting library (though not directly used in the final models here, it's often part of a comprehensive time series toolkit).

* **`sklearn.linear_model.LinearRegression` and `sklearn.metrics.r2_score`**: From Scikit-learn, for building and evaluating linear regression models.

* **`mlforecast` and `mlforecast.utils.PredictionIntervals`**: A library that provides a Scikit-learn-like API for time series forecasting, allowing the integration of various models and features.

* **`utilsforecast.feature_engineering.fourier` and `pipeline`**: Functions to generate Fourier series features for capturing seasonality.

* **`scipy.stats`**: For statistical functions, potentially used in custom utilities.

In [ ]:
%load_ext autoreload
%autoreload 2

from functools import partial
import polars as pl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import MSTL, AutoARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape, mase, mse, smape
from plotting_utils import (
    plotly_series as plot_series,
    plot_residuals_diagnostic,
    plot_real_data_vs_insample_forecast,
)
from summary_utils import (
    print_arima_fitted_summary,
    print_regression_summary_from_model,
    get_fitted_residuals,
)

from prophet import Prophet  # Imported but not directly used in this version

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

from utilsforecast.feature_engineering import fourier, pipeline
from scipy import stats

### 1.2. Global Configurations

We set the default Plotly template for consistent visual aesthetics and define a list of evaluation metrics we'll use throughout the notebook.

In [ ]:
pio.templates.default = "plotly_white"

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(
        mase, seasonality=48
    ),  # MASE requires a seasonality parameter, 48 for daily 30-min data
]

## 2. Data Loading and Initial Preparation

We load the preprocessed London Smart Meter data and perform essential transformations to get it into a format suitable for time series analysis and forecasting libraries.

### 2.1. Load Data and Standardize Column Names

The data is loaded from a parquet file. We then generate a complete `ds` (datetime) column for each `LCLid` to ensure all time points are present, and rename key columns (`LCLid` to `unique_id`, `energy_consumption` to `y`) to align with `statsforecast` and `mlforecast` conventions.

In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

### 2.2. Define Column Variables

For readability and ease of use, we define variables for our important column names.

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
temp_ = "temperature"  # Exogenous variable: temperature
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)
temp_col = pl.col(temp_)

### 2.3. Select Relevant Columns and Handle Missing Values

We filter the data to a specific block (`block_7`) and select only the columns relevant for our analysis, including various weather features and household demographics. We then `explode` the list columns to create individual rows for each time step. Finally, for our selected time series, we forward-fill and backward-fill any missing values in the target variable (`y`) and temperature to ensure a continuous series.

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_,
            "Acorn",
            "Acorn_grouped",
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            "holidays",
            "visibility",
            "windBearing",
            "temperature",
            "dewPoint",
            "pressure",
            "apparentTemperature",
            "windSpeed",
            "precipType",
            "icon",
            "humidity",
            "summary",
        ]
    )
)
data.head()

# Select a single meter for detailed analysis and ensure no missing values
selected_id = "MAC000193"
data = (
    data.filter(id_col.eq(selected_id))
    .with_columns(
        target_col.forward_fill().backward_fill()
    )  # Handle missing target values
    .select(
        [time_, id_, target_, temp_]
    )  # Select only relevant columns for this analysis
)
data.head()

## 3. Exploratory Data Analysis (EDA)

Before modeling, it's crucial to visualize our time series data and understand the relationship between the target variable (electricity consumption) and our exogenous variable (temperature).

### 3.1. Visualize Electricity Consumption

This plot shows the electricity consumption over time for our selected meter. We expect to see strong daily and annual seasonal patterns.

In [ ]:
plot_series(data)

### 3.2. Visualize Temperature

This plot shows the temperature over time. Temperature is a key driver of electricity demand, especially for heating and cooling.

In [ ]:
plot_series(data, target_col=temp_)

### 3.3. Relationship between Consumption and Temperature

This scatter plot is critical for understanding how electricity consumption (`y`) changes with temperature. For residential electricity demand, we often observe a **U-shaped or V-shaped relationship**:
* **Low Temperatures:** High consumption (due to heating).
* **Moderate Temperatures:** Lower consumption (less heating/cooling needed).
* **High Temperatures:** High consumption (due to air conditioning).

This non-linear relationship implies that a simple linear model might not fully capture the effect of temperature, and we might need to consider transformations or more complex models later.

In [ ]:
px.scatter(
    data,
    y=target_,
    x=temp_,
    title="Electricity Consumption vs. Temperature",
    labels={target_: "Electricity Consumption", temp_: "Temperature"},
    template="plotly_white",
).show()

## 4. Baseline Model: Linear Regression with Temperature

We start with a simple Linear Regression model to establish a baseline. This model will use temperature as the sole predictor for electricity consumption. We'll use `MLForecast` to facilitate this.

### 4.1. Fit the Linear Regression Model

`MLForecast` allows us to define a model (here, `LinearRegression`) and fit it to our time series data. The `fitted=True` argument ensures that in-sample predictions are stored, which is useful for residual analysis.

In [ ]:
# Initialize MLForecast with LinearRegression model
mf = MLForecast(models=LinearRegression(), freq="30min")

# Convert data to pandas DataFrame for MLForecast compatibility
data_pd = data.to_pandas()

# Fit model. We specify static_features=[] as temperature is a time-varying exogenous feature.
mf.fit(data_pd, fitted=True, static_features=[])

### 4.2. Access Model and In-Sample Forecasts

After fitting, we can access the underlying `LinearRegression` model and retrieve the in-sample forecasts (predictions made on the training data itself).

In [ ]:
model = mf.models_["LinearRegression"]
insample_forecasts = mf.forecast_fitted_values()  # Get in-sample predictions
X = data_pd.select(
    model.feature_names_in_
).to_pandas()  # Extract features used by the model
y = data_pd.get_column(target_).to_pandas()  # Extract actual target values

### 4.3. Print Regression Summary

This summary provides key statistics about the fitted linear regression model:
* **Coefficients:** The estimated impact of each predictor (temperature) on electricity consumption.
* **R-squared:** The proportion of variance in the target variable that is explained by the model. A higher R-squared indicates a better fit.
* **P-values:** For each coefficient, indicating its statistical significance.

In [ ]:
print_regression_summary_from_model(model, X, y)

### 4.4. Visualize In-Sample Forecasts vs. Actuals

These plots help us visually assess how well the linear regression model fits the historical data.

#### 4.4.1. Time Series Plot of Actuals vs. Forecasts

This plot overlays the model's predictions on the actual electricity consumption over time. We can see if the model captures the overall level and some variations, but it will likely miss the strong seasonal patterns if only temperature is used.

In [ ]:
plot_series(
    data, insample_forecasts.drop("y")
)  # Drop 'y' from insample_forecasts as it's already in 'data'

#### 4.4.2. Scatter Plot of Real vs. In-sample Forecast

An ideal forecast would lie perfectly on the `y=x` line. This scatter plot helps visualize the spread of predictions around the actual values. Deviations from the diagonal line indicate prediction errors.

In [ ]:
px.scatter(
    x=insample_forecasts.get_column("y"),
    y=insample_forecasts.get_column("LinearRegression"),
).update_traces(marker=dict(size=5)).update_layout(
    title="Real vs In-sample Forecast",
    xaxis_title="Real Electricity Consumption",
    yaxis_title="In-sample Forecast (Linear Regression)",
    template="plotly_white",
).show()

## 5. Residual Analysis for Linear Regression

Residuals are the differences between the actual values and the model's predictions. Analyzing residuals is crucial for understanding if our model has captured all the relevant information and if its assumptions are met. Ideally, residuals should be "white noise" (random, no patterns).

### 5.1. Extract Residuals

In [ ]:
residuals = get_fitted_residuals(mf)
residuals = residuals.get_column(
    "LinearRegression"
)  # Assuming 'LinearRegression' is the model name
ds = data.get_column(time_)

### 5.2. Residual Diagnostics Plot

This plot provides a comprehensive view of the residuals:
* **Time Series Plot of Residuals:** Should show no patterns, trends, or changing variance.
* **ACF Plot of Residuals:** Should show no significant spikes (autocorrelation) at any lags, indicating that the model has captured all temporal dependencies.
* **PACF Plot of Residuals:** Similar to ACF, should show no significant partial autocorrelation.
* **Histogram of Residuals:** Should ideally be normally distributed around zero.

Given that our linear regression only uses temperature, we expect to see strong seasonal patterns remaining in the residuals, as the model hasn't explicitly accounted for them.

In [ ]:
plot_residuals_diagnostic(
    residuals=residuals,
    time=ds,
)

### 5.3. Ljung-Box Test on Residuals

The Ljung-Box test formally checks for autocorrelation in the residuals.
* **Null Hypothesis (H0):** The residuals are independently distributed (i.e., they are white noise).
* **Alternative Hypothesis (H1):** The residuals are not independently distributed (i.e., there is significant autocorrelation).
* **Interpretation:** A low p-value (e.g., < 0.05) indicates that the residuals are *not* white noise, meaning our model has missed some patterns. We expect a low p-value here, confirming the visual patterns in the residual plots.

In [ ]:
lb_test_results = acorr_ljungbox(residuals, lags=[10, 48, 96])  # Check common lags
print("Ljung-Box Test Results on Linear Regression Residuals:")
print(lb_test_results)

print("\nInterpretation:")
for i, lag in enumerate(lb_test_results["lb_pvalue"].index):
    p_value = lb_test_results["lb_pvalue"].iloc[i]
    if p_value > 0.05:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals appear to be white noise (Fail to reject H0)."
        )
    else:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals are NOT white noise (Reject H0). This indicates uncaptured autocorrelation."
        )

### 5.4. Scatter Plot of Residuals vs. Temperature

This plot helps us identify if there's any remaining non-linear relationship between temperature and the error terms. If the linear model didn't fully capture the U-shaped relationship, you might see a pattern here (e.g., a curved shape).

In [ ]:
fig = px.scatter(x=data.get_column(temp_), y=residuals)
fig.update_layout(
    title="Scatter Plot of Residuals vs. Temperature",
    xaxis_title="Temperature",
    yaxis_title="Residuals",
    template="plotly_white",
    width=800,
    height=600,
    showlegend=False,
).show()

### 5.5. Scatter Plot of Residuals vs. Fitted Values

This plot helps check for **heteroscedasticity** (non-constant variance of residuals) or other systematic errors. Ideally, residuals should be randomly scattered around zero, with no fanning-out or funneling patterns.

In [ ]:
fig = px.scatter(x=insample_forecasts.get_column("LinearRegression"), y=residuals)
fig.update_layout(
    title="Scatter Plot of Residuals vs. Fitted Values",
    xaxis_title="Fitted Values (Linear Regression)",
    yaxis_title="Residuals",
    template="plotly_white",
    width=800,
    height=600,
    showlegend=False,
).show()

### 5.6. Evaluate In-Sample Performance

We quantify the in-sample performance of our baseline linear regression model using the defined metrics.

In [ ]:
evaluate(
    insample_forecasts,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

## 6. Cross-Validation for Robust Evaluation

In time series, a simple train-test split might not be robust enough. **Cross-validation** (specifically, rolling origin or walk-forward validation) provides a more reliable estimate of out-of-sample performance by simulating multiple forecast scenarios over different time windows.

### 6.1. Perform Cross-Validation

`MLForecast`'s `cross_validation` method allows us to perform this.
* `h`: The forecast horizon (e.g., 48\*7 for 7 days).
* `step_size`: How many steps to advance the training window for each new forecast.
* `n_windows`: The number of forecast windows to create.

In [ ]:
horizon = 48 * 7  # 7 days ahead forecast
y_hat = mf.cross_validation(
    df=data.select([id_, time_, target_, temp_]).to_pandas(),
    h=horizon,
    step_size=1,  # Advance one step at a time
    n_windows=1,  # Create only one window for simplicity in this example
    fitted=True,
    static_features=[],
).drop(columns=["cutoff"])  # 'cutoff' column is not needed for evaluation

### 6.2. Evaluate Cross-Validation Performance

The metrics calculated here provide a more realistic assessment of how the model would perform on new, unseen data.

In [ ]:
evaluate(
    pl.from_pandas(y_hat),  # Convert back to Polars DataFrame for evaluation
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

### 6.3. Visualize Cross-Validation Forecasts

This plot shows the actual data and the cross-validated forecasts, giving a visual sense of the model's predictive capability on "future" data.

In [ ]:
plot_series(
    data, pl.from_pandas(y_hat), max_insample_length=horizon
)  # Plot actuals and cross-validated forecasts

## 7. Enhancing Regression with Fourier Series for Seasonality

The previous linear regression model likely struggled with capturing the complex daily and annual seasonality. **Fourier series** are a powerful way to represent periodic patterns using sine and cosine waves of different frequencies. By adding these as features, a linear model can then capture non-linear seasonal effects.

### 7.1. Generate Fourier Features

We use `utilsforecast.feature_engineering.fourier` to create sine and cosine terms for different seasonal periods:
* `season_length=2*24` (48): Daily seasonality (30-minute data).
* `season_length=2*24*7` (336): Weekly seasonality.
* `season_length=2*24*365` (17520): Annual seasonality.
* `k`: The number of sine and cosine pairs (harmonics) to include. Higher `k` captures more complex shapes but also increases model complexity.

The `pipeline` function applies these feature transformations and also generates future values for these features, which are needed for forecasting.

In [ ]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data,
    features=features,
    freq="30m",
    h=horizon,  # Horizon for future features
)

Let's inspect the `data_fourier` to see the newly added columns.

In [ ]:
data_fourier.head()

### 7.2. Fit Linear Regression with Fourier Features

Now, we re-fit the `LinearRegression` model using the `data_fourier` which includes our temperature and the newly generated Fourier features.

In [ ]:
mf.fit(data_fourier, fitted=True, static_features=[])

### 7.3. Print Regression Summary (with Fourier)

Observe how the R-squared value has likely increased significantly, and you'll see coefficients for all the Fourier terms, indicating their contribution to explaining the electricity demand.

In [ ]:
model = mf.models_["LinearRegression"]
insample_forecasts = mf.forecast_fitted_values()
X = data_fourier.select(model.feature_names_in_).to_pandas()
y = data_fourier.get_column(target_).to_pandas()

print_regression_summary_from_model(model, X, y)

### 7.4. Visualize In-Sample Forecasts (with Fourier)

You should see a much better fit to the actual data, with the model now capable of capturing the strong seasonal fluctuations.

In [ ]:
plot_series(data, insample_forecasts.drop("y"))

In [ ]:
px.scatter(
    x=insample_forecasts.get_column("y"),
    y=insample_forecasts.get_column("LinearRegression"),
).update_traces(marker=dict(size=5)).update_layout(
    title="Real vs In-sample Forecast (with Fourier Features)",
    xaxis_title="Real Electricity Consumption",
    yaxis_title="In-sample Forecast (Linear Regression + Fourier)",
    template="plotly_white",
).show()

## 8. Residual Analysis for Regression with Fourier Features

Let's re-examine the residuals after adding Fourier features. We expect a significant improvement, with fewer patterns remaining.

### 8.1. Extract Residuals

In [ ]:
residuals = get_fitted_residuals(mf)
residuals = residuals.get_column("LinearRegression")
ds = data_fourier.get_column(time_)  # Use data_fourier's time column

### 8.2. Residual Diagnostics Plot (with Fourier)

The ACF and PACF plots of the residuals should now show much less autocorrelation, especially at seasonal lags. The time plot should appear more like white noise.

In [ ]:
plot_residuals_diagnostic(
    residuals=residuals,
    time=ds,
)

### 8.3. Ljung-Box Test on Residuals (with Fourier)

We expect higher p-values from the Ljung-Box test, indicating that the residuals are closer to white noise. However, there might still be some remaining autocorrelation, especially at lower lags, which a pure regression model doesn't explicitly capture.

In [ ]:
lb_test_results = acorr_ljungbox(residuals, lags=[10, 48, 96])
print("Ljung-Box Test Results on Linear Regression Residuals (with Fourier Features):")
print(lb_test_results)

print("\nInterpretation:")
for i, lag in enumerate(lb_test_results["lb_pvalue"].index):
    p_value = lb_test_results["lb_pvalue"].iloc[i]
    if p_value > 0.05:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals appear to be white noise (Fail to reject H0)."
        )
    else:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals are NOT white noise (Reject H0). This indicates uncaptured autocorrelation."
        )

### 8.4. Scatter Plot of Residuals vs. Temperature (with Fourier)

If the Fourier terms effectively captured the non-linear seasonal patterns, and temperature's effect is now better modeled, this plot should show a more random scatter.

In [ ]:
fig = px.scatter(x=data_fourier.get_column(temp_), y=residuals)
fig.update_layout(
    title="Scatter Plot of Residuals vs. Temperature (with Fourier Features)",
    xaxis_title="Temperature",
    yaxis_title="Residuals",
    template="plotly_white",
    width=800,
    height=600,
    showlegend=False,
).show()

### 8.5. Scatter Plot of Residuals vs. Fitted Values (with Fourier)

This plot should also show a more random scatter, indicating improved homoscedasticity (constant variance of residuals).

In [ ]:
fig = px.scatter(x=insample_forecasts.get_column("LinearRegression"), y=residuals)
fig.update_layout(
    title="Scatter Plot of Residuals vs. Fitted Values (with Fourier Features)",
    xaxis_title="Fitted Values (Linear Regression + Fourier)",
    yaxis_title="Residuals",
    template="plotly_white",
    width=800,
    height=600,
    showlegend=False,
).show()

### 8.6. Evaluate In-Sample Performance (with Fourier)

Compare these metrics to the baseline model. You should see a significant improvement across most metrics.

In [ ]:
evaluate(
    insample_forecasts,
    metrics=metrics,
    train_df=data_fourier.select([id_, time_, target_]),
)

## 9. Cross-Validation for Regression with Fourier Features

Let's perform cross-validation again with the enhanced model to get a robust estimate of its out-of-sample performance.

In [ ]:
mf = MLForecast(models=LinearRegression(), freq="30min")

y_hat = mf.cross_validation(
    df=data_fourier,  # Use the data with Fourier features
    h=horizon,
    step_size=1,
    n_windows=1,
    fitted=True,
    static_features=[],
).drop("cutoff")

### 9.1. Evaluate Cross-Validation Performance (with Fourier)

In [ ]:
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data_fourier,
)

### 9.2. Visualize Cross-Validation Forecasts (with Fourier)

In [ ]:
plot_series(data, y_hat, max_insample_length=horizon)

## 10. Dynamic Regression: Combining ARIMA with Exogenous Variables (SARIMAX)

Even after adding Fourier series, there might still be some remaining autocorrelation in the residuals (as indicated by the Ljung-Box test). This is where **Dynamic Regression**, often implemented as a **SARIMAX (Seasonal AutoRegressive Integrated Moving Average with eXogenous regressors)** model, comes in.

The idea is to:

1. Use a regression component (e.g., Linear Regression with temperature and Fourier features) to capture the deterministic parts of the series (trend, seasonality, and exogenous variable effects).

2. Then, use an ARIMA model to capture the remaining autocorrelation structure in the *residuals* of that regression.

In `statsforecast`, you can achieve this by fitting an `AutoARIMA` model to the data that already contains the exogenous features. The `AutoARIMA` model will then effectively act as the ARMA component on the residuals, while the linear model (implicitly handled by the `data_fourier` input) acts as the regression part.

Here, we're fitting an `AutoARIMA` model directly to `data_fourier`. By setting `max_d=0` and `seasonal=False`, we are explicitly telling `AutoARIMA` to find only non-seasonal ARMA components (`p`, `q`) on the series *after* the Fourier terms have accounted for seasonality and any overall trend. If you wanted the ARIMA part to also handle differencing or seasonality, you would adjust these parameters.

In [ ]:
sf = StatsForecast(
    models=[AutoARIMA(max_d=0, seasonal=False, nmodels=20, max_p=3, max_q=3)],
    freq="30m",
    n_jobs=-1,  # Use all available cores
)

# Fit AutoARIMA to the data with Fourier features.
# AutoARIMA will internally use the Fourier features as exogenous regressors.
sf.fit(data_fourier.to_pandas())  # Convert to pandas for StatsForecast

### 10.1. Print ARIMA Fitted Summary

This summary will show the selected `(p,q)` orders for the ARIMA component that models the residuals.

In [ ]:
print_arima_fitted_summary(sf.fitted_[0, 0].model_)

### 10.2. Residual Analysis for Dynamic Regression

We expect the residuals from this combined model to be very close to white noise, as both the deterministic (regression + Fourier) and stochastic (ARIMA) components have been modeled.

In [ ]:
residuals = sf.fitted_[0, 0].model_["residuals"]
time = data_fourier.get_column(time_)

plot_residuals_diagnostic(
    residuals=residuals,
    time=time,
)

### 10.3. Ljung-Box Test for Dynamic Regression Residuals

Ideally, the p-values for the Ljung-Box test should now be high (e.g., > 0.05), indicating that the residuals are indeed white noise and the model has captured almost all the information.

In [ ]:
lb_test_results = acorr_ljungbox(residuals, lags=[10, 48, 96])
print("Ljung-Box Test Results on Dynamic Regression Residuals:")
print(lb_test_results)

print("\nInterpretation:")
for i, lag in enumerate(lb_test_results["lb_pvalue"].index):
    p_value = lb_test_results["lb_pvalue"].iloc[i]
    if p_value > 0.05:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals appear to be white noise (Fail to reject H0)."
        )
    else:
        print(
            f"  At lag {lag}: P-value = {p_value:.3f}. Residuals are NOT white noise (Reject H0). This indicates uncaptured autocorrelation."
        )

## Conclusion

This notebook has guided you through building increasingly sophisticated time series forecasting models for electricity demand:

1. **Baseline Linear Regression:** We started with a simple linear model using temperature, highlighting its limitations in capturing seasonality and autocorrelation.

2. **Enhanced Regression with Fourier Series:** We significantly improved the model's performance by incorporating Fourier series to explicitly model complex daily, weekly, and annual seasonal patterns. This demonstrated the power of feature engineering for time series.

3. **Dynamic Regression (SARIMAX Concept):** Finally, we showed how an ARIMA component can be combined with a regression model (implicitly via `AutoARIMA` on data with exogenous features) to capture any remaining autocorrelation in the residuals, leading to a more complete and accurate model.

By combining the strengths of regression (for exogenous variables and deterministic patterns) and ARIMA (for stochastic dependencies), you can build powerful and accurate forecasting systems for complex time series data like electricity demand. Remember that the "best" model often involves iterative analysis, careful feature selection, and thorough residual diagnostics.